# Data Merging — Stage 3 Cleaning 06: Which Factors Dropped

## Input
- `Data/Data_Collection/Final/Stage_3_Cleaning/exclusion_manifest.csv` (from notebook 01)
- `Data/Data_Collection/Final/Stage_4_Normalised/decisions_aggregate.csv` (from notebook 04)
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/decisions_panel.csv` (from notebook 05)
- `Data/Data_Collection/Final/Stage_4_Normalised/union_drop_list.csv` (from notebook 05)
- `Data/Data_Collection/Final/Stage_4_Normalised/floor_diagnostic.csv` (used in the final ad-hoc cell)
- `lib.review` — shared constants (`STRUCTURAL`, `RULE_ORDER`, `RULE_LABEL`)
- A hardcoded `MANUAL` set mirroring `MANUAL_DROP` from `Stage_4_Assembly/01_apply_union.ipynb`

## Purpose
A **read-only reconciliation notebook**. It doesn't drop, transform, or save any feature data — its sole job is to trace every removal decision made across the entire Stage 3 pipeline (a priori exclusions, measured statistical rules, manual drops, and structural identity-enforcement) back to a single, auditable source of truth, and to report everything at **base-factor level** rather than column level.

## Why Base-Factor Level Matters
The full-moments tables carry five columns per stock factor (`_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread`). Counting drops at the column level would count a single failed factor five times over in those tables. This notebook consistently restricts counts to the means-only tables (`MEANS = {agg_market_daily_means, agg_market_monthly_means, weekly_raw}`), which hold exactly one column per base factor, whenever a base-factor total is being computed.

## Cell 1 — Load Everything
Loads the four CSV sources (manifest, aggregate decisions, panel decisions, union drop list), reporting row counts or a missing-file warning for each. Also defines `MANUAL`, a hardcoded mirror of the `MANUAL_DROP` set used later in `Stage_4_Assembly/01_apply_union.ipynb` — included here purely so this reconciliation notebook can account for those drops too, even though the actual removal logic lives elsewhere.

## Cell 2 — A Priori Exclusions
Reports the two exclusion channels that are **written-down judgements rather than measurements**:
- The Stage 3 manifest (notebook 01's exclusion list), broken down by category if a category column is present.
- The structural list (`rv.STRUCTURAL`), broken down by category.

Checks the intersection between the manifest and the structural list — expected to be disjoint ("as intended") since these are meant to be two separate a priori channels, not overlapping ones.

## Cell 3 — Measured Rules, at Base-Factor Level
Filters `decisions_aggregate.csv` to only the means-tables rows with `action == 'drop'` (`ab`), and `decisions_panel.csv` to all dropped rows (`pb`, since the panel has no moments to worry about). Builds two views:
- **Primary reason** — the first rule to fire, in precedence order, deduplicated per base factor (these partition the dropped set, since each factor has exactly one primary reason).
- **All rules fired** — for each rule, counts how many distinct base factors had that rule appear *anywhere* in their `all_rules` field, noting explicitly that a factor can appear under several rules so these counts won't sum to the primary-reason totals.

Reports totals separately for aggregate, panel, and the union of both.

## Cell 4 — Moment-Only Column Drops
Isolates the subset of full-moments-table column drops where the **base factor's cap-weighted mean survived** — meaning only a higher moment (`_cwstd`, `_cwskew`, `_cwkurt`, `_spread`) failed. These are flagged as **aggregate-only, with no base-factor or panel equivalent**, and the cell explicitly warns that they must never be folded into base-factor drop totals elsewhere in the notebook. Breaks these down by which rule fired and by which moment suffix was affected.

## Cell 5 — Identity Enforcement
Compares the set of base factors that carry a `_cwmean` column in the aggregate (i.e., stock factors that made it into the aggregate pipeline) against the full set of base factors present in the panel. Computes:
- **Panel only** — factors present in the panel but never aggregated (so the union mechanism from notebook 05 has nothing to remove them with, since removing a panel column the aggregate never held is a no-op for the aggregate).
- **Aggregate only** — the reverse.

These asymmetric leftovers are exactly what required an explicit "identity enforcement" pass in `Stage_4_Assembly/01` — a manual cleanup step to force both pipelines onto the same base-factor set, since the automated union logic from notebook 05 can only reconcile factors that exist in *both* places.

## Cell 6 — Full Reconciliation Table
Assembles a single summary table (`chan`) listing every removal channel and its base-factor count:
1. Stage 1/1.5/2 upstream drops (explicitly out of scope — "NOT COVERED HERE").
2. Stage 3 exclusion manifest.
3. Structural list.
4. Measured rules R0–R6 (minus anything already counted under structural, to avoid double-counting).
5. The collated union (from `union_drop_list.csv`).
6. Manual drops (imputation + Stage 2 asymmetry corrections).
7. Identity enforcement.

Then reports two extra diagnostics:
- **Composition of the union** — how much of the final union drop list is already explained by the structural list vs. measured purely by the statistical rules.
- **Convergent validity** — the overlap between the structural list (written from construction logic *before* any z-scores were computed) and the measured rules (which fired independently, based on actual data). Agreement between the two is treated as evidence that both channels are catching real problems, since they were derived independently.

## Cell 7 — Write `all_drops.csv`
Builds one long-form table with a row per (base factor, channel) pair, covering manifest exclusions, structural exclusions, every rule-fired drop in both pipelines, manual drops, and both directions of identity enforcement. Reports how many base factors were caught by more than one channel (redundant confirmation across independent methods), and saves the result.

## Post-Cell-7 Ad-Hoc Cells (uncommented, exploratory)
Three additional cells extend the reconciliation without being formally numbered:

1. **Rule-trip breakdown by three blocks** (`macro`, `stock_aggregate`, `stock_panel`) — for each block, counts how many base factors tripped *each* rule at all (not just as primary reason), plus a total dropped-factor count per block. Macro factors are separated from stock-aggregate factors by checking membership in the `_cwmean`-bearing stock set.
2. **Rule-agreement check for shared stock factors** — for every base factor dropped in *both* the aggregate and panel pipelines, prints the primary rule that fired in each, flagging any case where the two pipelines disagreed on *why* a factor failed even though both agreed it should be dropped.
3. **Floor diagnostic cross-reference** — loads `floor_diagnostic.csv`, filters to daily-table features with a nonzero epsilon-bound count (`n_eps_bound > 0`), and merges against the aggregate decisions to show, for each such feature, whether it was ultimately dropped and by which rule — a targeted check on whether the robust-scale floor mechanism specifically correlates with drop decisions.

## Output
- `Data/Data_Collection/Final/Stage_4_Normalised/all_drops.csv` — one row per (base factor, removal channel) pair, with `kind` (a priori / measured / manual / structural), `rule`, and `pipeline` (aggregate / panel / both) columns. This is the single canonical file answering "why was factor X removed, and from where."

No other files are written; every other output in this notebook is printed for inspection only.

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# WHICH FACTORS WERE DROPPED, AND BY WHAT
#
# Read-only. Reconciles every removal channel from the Stage 3 manifest through
# to the final model-ready column counts, and breaks the measured rules down at
# BASE FACTOR level rather than column level.
#
# Code/Data_Merging/Stage_3_Cleaning/06_which_factors_dropped.ipynb
# ═══════════════════════════════════════════════════════════════════════════════

# ── CELL 1 ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append('../..')
import lib.review as rv

pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 400)

ROOT    = Path('../../../Data/Data_Collection/Final')
STAGE3  = ROOT / 'Stage_3_Cleaning'
AGG_Z   = ROOT / 'Stage_4_Normalised'
PANEL_Z = ROOT / 'Stage_4_Normalised_Panel'
UNIONED = ROOT / 'Stage_5_Model_Ready' / '01_unioned'

# One column per base factor in these; the full-moments tables repeat each
# stock factor five times, so base-factor counts must come from here.
MEANS = {'agg_market_daily_means', 'agg_market_monthly_means', 'weekly_raw'}
MOM   = {'agg_market_daily_full_moments', 'agg_market_monthly_full_moments'}

# From Stage_4_Assembly/01_apply_union.ipynb MANUAL_DROP
MANUAL = {'stock_skew_chg_5d'} | {
    'CBOperProf', 'NetPayoutYield', 'AOP', 'GrSaleToGrOverhead', 'GP',
    'MomSeason16YrPlus', 'GrSaleToGrInv', 'MomOffSeason16YrPlus',
    'DivYieldST', 'fgr5yrLag', 'MomSeason11YrPlus', 'InvestPPEInv'}

def load(p, label):
    if p.exists():
        return pd.read_csv(p)
    print(f'  !! {label} not found at {p}')
    return None

man = load(STAGE3  / 'exclusion_manifest.csv', 'exclusion_manifest.csv')
a   = load(AGG_Z   / 'decisions_aggregate.csv', 'decisions_aggregate.csv')
p   = load(PANEL_Z / 'decisions_panel.csv',     'decisions_panel.csv')
uni = load(AGG_Z   / 'union_drop_list.csv',     'union_drop_list.csv')

print('=' * 100)
print('SOURCES LOADED')
print('=' * 100)
for lbl, d in [('exclusion_manifest', man), ('decisions_aggregate', a),
               ('decisions_panel', p), ('union_drop_list', uni)]:
    print(f'  {lbl:<22} {"-" if d is None else f"{len(d):>6,} rows"}')


# ── CELL 2 ────────────────────────────────────────────────────────────────────
# A PRIORI EXCLUSIONS: the manifest and the structural list.
# Both are written-down judgements, not measurements, so they are reported
# separately from the rules.

print('\n' + '=' * 100)
print('A PRIORI EXCLUSIONS')
print('=' * 100)

man_set = set()
if man is not None:
    fcol = next((c for c in ('feature', 'factor', 'column', 'name')
                 if c in man.columns), None)
    man_set = set(man[fcol].astype(str))
    print(f'\n  STAGE 3 MANIFEST -- {len(man_set)} base factors')
    ccol = next((c for c in ('category', 'reason_category', 'group')
                 if c in man.columns), None)
    if ccol:
        print(man[ccol].value_counts().to_string())
    else:
        print(f'    (no category column; columns are {list(man.columns)})')

struct = {f: v[0] for f, v in rv.STRUCTURAL.items()}
print(f'\n  STRUCTURAL LIST -- {len(struct)} base factors')
print(pd.Series(list(struct.values())).value_counts().to_string())

overlap = man_set & set(struct)
print(f'\n  manifest ∩ structural : {len(overlap)}'
      + (f'  -> {sorted(overlap)}' if overlap else '   (disjoint, as intended)'))


# ── CELL 3 ────────────────────────────────────────────────────────────────────
# MEASURED RULES, AT BASE FACTOR LEVEL.
# The published tables count COLUMNS, so a stock factor failing in the
# full-moments table is counted five times. Counting base factors instead
# uses the means tables, which hold exactly one column per base factor.

print('\n' + '=' * 100)
print('MEASURED RULES -- BASE FACTORS (not columns)')
print('=' * 100)

ab = a[a['table_source'].isin(MEANS) & (a['action'] == 'drop')]
pb = p[p['action'] == 'drop']

def by_rule(df, col='rule_fired'):
    return df.drop_duplicates('base_factor')[col].value_counts()

tbl = pd.DataFrame({'aggregate': by_rule(ab), 'panel': by_rule(pb)})
tbl = tbl.fillna(0).astype(int)
tbl['total'] = tbl.sum(axis=1)
tbl = tbl.reindex([r for r in rv.RULE_ORDER if r in tbl.index])
tbl.index = [rv.RULE_LABEL.get(i, i) for i in tbl.index]
print('\n  PRIMARY reason (first rule in precedence order -- these partition):')
print(tbl.to_string())
print(f'\n  {"unique base factors dropped":<44} '
      f'{ab["base_factor"].nunique():>9} {pb["base_factor"].nunique():>6} '
      f'{len(set(ab["base_factor"]) | set(pb["base_factor"])):>6}')

# every rule that fired, not just the primary one
print('\n  ALL rules fired (a factor may appear under several):')
rows = []
for rule in rv.RULE_ORDER:
    na = ab[ab['all_rules'].str.contains(rule, regex=False, na=False)]['base_factor'].nunique()
    npn = pb[pb['all_rules'].str.contains(rule, regex=False, na=False)]['base_factor'].nunique()
    rows.append({'rule': rv.RULE_LABEL.get(rule, rule),
                 'aggregate': na, 'panel': npn, 'total': na + npn})
print(pd.DataFrame(rows).to_string(index=False))


# ── CELL 4 ────────────────────────────────────────────────────────────────────
# MOMENT-ONLY DROPS: full-moments columns removed where the base factor's
# cap-weighted mean survived. These are aggregate-only and have no base-factor
# equivalent, so they must never be added to the base-factor totals.

print('\n' + '=' * 100)
print('MOMENT-ONLY COLUMN DROPS  (aggregate only, no panel counterpart)')
print('=' * 100)

mom_drop = a[a['table_source'].isin(MOM) & (a['action'] == 'drop')]
mom_only = mom_drop[~mom_drop['base_factor'].isin(ab['base_factor'])]

print(f'\n  moment columns dropped in total      : {len(mom_drop)}')
print(f'  of which the base factor also went   : {len(mom_drop) - len(mom_only)}')
print(f'  MOMENT-ONLY (base factor survived)   : {len(mom_only)}')

if len(mom_only):
    print('\n  by rule:')
    print(mom_only['rule_fired'].map(lambda r: rv.RULE_LABEL.get(r, r))
                                .value_counts().to_string())
    print('\n  by moment suffix:')
    suf = mom_only['feature'].str.extract(
        r'(_cwmean|_cwstd|_cwskew|_cwkurt|_spread)$')[0]
    print(suf.value_counts().to_string())
    print(f'\n  affecting {mom_only["base_factor"].nunique()} distinct base factors')


# ── CELL 5 ────────────────────────────────────────────────────────────────────
# IDENTITY ENFORCEMENT: base factors that existed in one pipeline only.
# The union cannot remove a panel column the aggregate never held, so these
# were removed by an explicit pass in Stage_4_Assembly/01.

print('\n' + '=' * 100)
print('IDENTITY ENFORCEMENT  (present in one pipeline only)')
print('=' * 100)

# stock factors in the aggregate = those with a _cwmean column
agg_stock = set(a.loc[a['feature'].str.endswith('_cwmean'), 'base_factor'])
pan_all   = set(p['base_factor'])

panel_only = sorted(pan_all - agg_stock)
agg_only   = sorted(agg_stock - pan_all)

print(f'\n  aggregate stock base factors : {len(agg_stock)}')
print(f'  panel base factors           : {len(pan_all)}')
print(f'  shared                       : {len(agg_stock & pan_all)}')
print(f'\n  panel only     ({len(panel_only):>2}): {panel_only}')
print(f'  aggregate only ({len(agg_only):>2}): {agg_only}')
print(f'\n  removed by identity enforcement: {len(panel_only) + len(agg_only)}')


# ── CELL 6 ────────────────────────────────────────────────────────────────────
# FULL RECONCILIATION

print('\n' + '=' * 100)
print('EVERY REMOVAL CHANNEL')
print('=' * 100)

union_set = set(uni['base_factor']) if uni is not None else set()
rule_base = set(ab['base_factor']) | set(pb['base_factor'])
ident     = set(panel_only) | set(agg_only)

chan = pd.DataFrame([
    {'channel': 'Stage 1 / 1.5 / 2  (upstream)', 'kind': 'various',
     'base_factors': np.nan, 'source': 'data-collection doc -- NOT COVERED HERE'},
    {'channel': 'Stage 3 exclusion manifest',    'kind': 'a priori',
     'base_factors': len(man_set), 'source': 'exclusion_manifest.csv'},
    {'channel': 'structural list',               'kind': 'a priori',
     'base_factors': len(struct), 'source': 'review.py STRUCTURAL'},
    {'channel': 'measured rules R0-R6',          'kind': 'measured',
     'base_factors': len(rule_base - set(struct)), 'source': 'decisions_*.csv'},
    {'channel': '  -> collated as the union',    'kind': 'measured',
     'base_factors': len(union_set), 'source': 'union_drop_list.csv'},
    {'channel': 'manual (imputation + Stage 2)', 'kind': 'manual',
     'base_factors': len(MANUAL), 'source': 'Stage_4_Assembly/01 MANUAL_DROP'},
    {'channel': 'identity enforcement',          'kind': 'structural',
     'base_factors': len(ident), 'source': 'Stage_4_Assembly/01 pass 2'},
])
print(chan.to_string(index=False))

print(f'\n  separately, {len(mom_only)} MOMENT-ONLY column drops in the aggregate')
print(f'  (not base factors -- do not add these to the column above)')

# how much of the union is structural vs measured
if union_set:
    print('\n' + '-' * 100)
    print('COMPOSITION OF THE 85-FACTOR UNION')
    print('-' * 100)
    print(f'  also on the structural list : {len(union_set & set(struct))}')
    print(f'  measured only               : {len(union_set - set(struct))}')

# convergent validity
conv = set(struct) & rule_base
print('\n' + '-' * 100)
print('CONVERGENT VALIDITY')
print('-' * 100)
print(f'  structural factors that ALSO failed a measured rule: '
      f'{len(conv)} of {len(struct)}  ({len(conv)/max(len(struct),1):.0%})')
print('  The structural list was written from construction logic before any')
print('  z-scores existed, so agreement is independent evidence for both.')
if conv:
    print(f'  {sorted(conv)}')


# ── CELL 7 ────────────────────────────────────────────────────────────────────
# ONE LONG-FORM FILE: every base factor ever removed, with the channel and
# reason attached.

print('\n' + '=' * 100)
print('WRITING all_drops.csv')
print('=' * 100)

recs = []
for f in sorted(man_set):
    recs.append({'base_factor': f, 'channel': 'manifest', 'kind': 'a priori',
                 'rule': '', 'pipeline': 'both'})
for f, cat in sorted(struct.items()):
    recs.append({'base_factor': f, 'channel': 'structural', 'kind': 'a priori',
                 'rule': cat, 'pipeline': 'both'})
for _, r in ab.drop_duplicates('base_factor').iterrows():
    recs.append({'base_factor': r['base_factor'], 'channel': 'rules',
                 'kind': 'measured', 'rule': r['rule_fired'], 'pipeline': 'aggregate'})
for _, r in pb.drop_duplicates('base_factor').iterrows():
    recs.append({'base_factor': r['base_factor'], 'channel': 'rules',
                 'kind': 'measured', 'rule': r['rule_fired'], 'pipeline': 'panel'})
for f in sorted(MANUAL):
    recs.append({'base_factor': f, 'channel': 'manual', 'kind': 'manual',
                 'rule': 'imputation / Stage 2 asymmetry', 'pipeline': 'both'})
for f in panel_only:
    recs.append({'base_factor': f, 'channel': 'identity', 'kind': 'structural',
                 'rule': 'panel only', 'pipeline': 'panel'})
for f in agg_only:
    recs.append({'base_factor': f, 'channel': 'identity', 'kind': 'structural',
                 'rule': 'aggregate only', 'pipeline': 'aggregate'})

alld = pd.DataFrame(recs)
alld.to_csv(AGG_Z / 'all_drops.csv', index=False)

print(f'  {len(alld)} rows, {alld["base_factor"].nunique()} distinct base factors')
print(f'\n  by channel:')
print(alld.groupby(['kind', 'channel']).size().to_string())
print(f'\n  base factors removed by more than one channel:')
multi = alld.groupby('base_factor')['channel'].nunique()
print(f'    {int((multi > 1).sum())} of {alld["base_factor"].nunique()}')
print(f'\n  saved -> {AGG_Z / "all_drops.csv"}')

SOURCES LOADED
  exclusion_manifest        250 rows
  decisions_aggregate     2,724 rows
  decisions_panel           354 rows
  union_drop_list            85 rows

A PRIORI EXCLUSIONS

  STAGE 3 MANIFEST -- 82 base factors
category
weekly_rebuild         68
redundant_dollar       60
redundant_weighting    36
redundant_venue        18
calendar               16
broken_taq             12
decided                12
sparse_indicator       12
degenerate_moment       6
level_superseded        6
zero_information        4

  STRUCTURAL LIST -- 31 base factors
cumulative_index         10
monotone_index_level      5
compound_ratio            4
venue_launch              3
product_launch            2
zero_denominator          2
coverage_diagnostic       1
deterministic_in_time     1
model_output              1
policy_step               1
column_invalid            1

  manifest ∩ structural : 0   (disjoint, as intended)

MEASURED RULES -- BASE FACTORS (not columns)

  PRIMARY reason (first rule in pr

In [3]:
a = pd.read_csv(AGG_Z / 'decisions_aggregate.csv')
p = pd.read_csv(PANEL_Z / 'decisions_panel.csv')

MEANS = {'agg_market_daily_means', 'agg_market_monthly_means', 'weekly_raw'}

# a stock factor is exactly one carrying a _cwmean column in a full-moments table
stock = set(a.loc[a['feature'].str.endswith('_cwmean'), 'base_factor'])

ab = a[a['table_source'].isin(MEANS) & (a['action'] == 'drop')].drop_duplicates('base_factor')
pb = p[p['action'] == 'drop'].drop_duplicates('base_factor')

blocks = {
    'macro':           ab[~ab['base_factor'].isin(stock)],
    'stock_aggregate': ab[ ab['base_factor'].isin(stock)],
    'stock_panel':     pb,
}

# every rule a factor tripped, not just the primary one -> columns will not
# sum to the totals, because a factor can appear under several rules
tbl = pd.DataFrame({
    name: {rule: int(df['all_rules'].str.contains(rule, regex=False, na=False).sum())
           for rule in rv.RULE_ORDER}
    for name, df in blocks.items()
})

tbl.index = [rv.RULE_LABEL.get(i, i) for i in tbl.index]
tbl.loc['— base factors dropped —'] = [len(df) for df in blocks.values()]
print(tbl.to_string())

                                                 macro  stock_aggregate  stock_panel
Structural (construction)                           18               13           13
R4  monotone trending level, has _mom/_yoy twin      8                0            0
R5  |rho| > 0.995 on first differences               5               14            9
R0  too little history in diag window                0                0            0
R1  modal_share > 0.90                               0                1            0
R2  std_of_z_ex_capped outside [0.5, 2.0]           10                8            2
R3  warm-up degenerate (sigma_f = eps)               0                1            0
R6  pct_gt5_ex_crisis > 2%                           6                7            9
— base factors dropped —                            37               39           30


In [4]:
a_stock = set(ab[ab['base_factor'].isin(stock)]['base_factor'])
p_stock = set(pb['base_factor'])

shared = sorted(a_stock & p_stock)
print(f'shared {len(shared)}')
for f in shared:
    ra = ab.loc[ab.base_factor == f, 'rule_fired'].iat[0]
    rp = pb.loc[pb.base_factor == f, 'rule_fired'].iat[0]
    flag = '' if ra == rp else '   <- different rule'
    print(f'  {f:<34} agg {ra:<20} panel {rp}{flag}')

shared 21
  BPEBM                              agg S_structural         panel S_structural
  EBM                                agg S_structural         panel S_structural
  EntMult                            agg S_structural         panel S_structural
  FirmAge                            agg S_structural         panel S_structural
  Leverage                           agg R5_duplicate_rho     panel R5_duplicate_rho
  PredictedFE                        agg S_structural         panel S_structural
  RDS                                agg S_structural         panel S_structural
  VarCF                              agg S_structural         panel S_structural
  bestofrdepth_share_tw_to_shrout    agg R5_duplicate_rho     panel R5_duplicate_rho
  close_vs_mid                       agg S_structural         panel S_structural
  csize_to_shrout                    agg R6_boundary_mass     panel R6_boundary_mass
  ivol_q                             agg R2_std_bounds        panel R2_std_bounds
  nop

In [5]:
d = pd.read_csv(AGG_Z / 'floor_diagnostic.csv')
eps_daily = d[(d.n_eps_bound > 0) & d.table_source.str.contains('daily')]

a = pd.read_csv(AGG_Z / 'decisions_aggregate.csv')
print(eps_daily[['table_source','feature','rung','sigma_f','n_eps_bound']]
      .merge(a[['table_source','feature','action','rule_fired','all_rules']],
             on=['table_source','feature'], how='left').to_string(index=False))

                 table_source       feature  rung      sigma_f  n_eps_bound action    rule_fired     all_rules
       agg_market_daily_means        ivol_q     1 1.000000e-08         4643   drop R2_std_bounds R2_std_bounds
agg_market_daily_full_moments ivol_q_cwmean     1 1.000000e-08         4643   drop R2_std_bounds R2_std_bounds
